# Correlação: Confiança do Modelo vs Distância de Features

Este notebook analisa a correlação entre:
1. **Confiança do modelo** na geração de máscaras de segmentação
2. **Distância de cosseno** no espaço de features do backbone ResNet50 em relação às amostras de treino

## Objetivo
Verificar se amostras com alta confiança do modelo estão mais próximas das amostras de treino no espaço de features.

## 1. Setup e Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("/home/luizluz/Documentos/multi-task-fcn")
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from shapely.geometry import box

# Project imports
from src.model import build_model, load_weights
from src.utils import normalize, get_device
from src.io_operations import get_image_metadata, load_args

# Set device
DEVICE = get_device()
print(f"Using device: {DEVICE}")

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
# Configuration paths
DATA_PATH = PROJECT_ROOT / "bioflore_data"
ITER_PATH = DATA_PATH / "v02" / "iter_020"

# GeoPackage files (all_labels_set from each region)
ALL_LABELS_PATHS = [
    ITER_PATH / "region_0" / "new_labels" / "all_labels_set.gpkg",
    ITER_PATH / "region_1" / "new_labels" / "all_labels_set.gpkg",
    ITER_PATH / "region_2" / "new_labels" / "all_labels_set.gpkg",
]

# Training labels
TRAIN_LABELS_PATH = DATA_PATH / "shapes" / "train_labels.shp"

# Model checkpoint
CHECKPOINT_PATH = ITER_PATH / "exp_deeplabv3_resnet50" / "checkpoint.pth.tar"

# Orthoimage paths (one per region)
ORTHO_PATHS = [
    PROJECT_ROOT / "matematica_industria_data" / "raw" / "geotiffs" / "Mosaic_BigPlot_03.tif",  # region_0
    PROJECT_ROOT / "matematica_industria_data" / "raw" / "geotiffs" / "Mosaic_BigPlot_07.tif",  # region_1
    PROJECT_ROOT / "matematica_industria_data" / "raw" / "geotiffs" / "Mosaic_BigPlot_11.tif",  # region_2
]

# Model parameters (from args.yaml)
INPUT_DIMENSION = 256  # Size of input to model
CROP_SIZE = 1024  # Size of patches to extract
NUM_CLASSES = 4
DROPOUT_RATE = 0.5
BATCH_NORM = True

print("Configuration loaded successfully!")
print(f"\nPaths:")
for i, path in enumerate(ALL_LABELS_PATHS):
    print(f"  Region {i}: {path.exists()} - {path}")
print(f"  Train labels: {TRAIN_LABELS_PATH.exists()} - {TRAIN_LABELS_PATH}")
print(f"  Checkpoint: {CHECKPOINT_PATH.exists()} - {CHECKPOINT_PATH}")

## 2. Carregar GeoDataFrames

Carregamos:
- `all_labels_set.gpkg` das 3 regiões (amostras de alta confiança da iter_001)
- `train_labels.shp` (dados de treino originais)

Em seguida, removemos do conjunto `all_labels` as amostras que já estão no conjunto de treino.

In [ ]:
# Load all_labels from each region and concatenate
all_labels_gdfs = []
for region_idx, path in enumerate(ALL_LABELS_PATHS):
    gdf = gpd.read_file(path)
    gdf['region_idx'] = region_idx
    all_labels_gdfs.append(gdf)
    print(f"Region {region_idx}: {len(gdf)} polygons loaded from {path.name}")

# Concatenate all regions
all_labels_gdf = pd.concat(all_labels_gdfs, ignore_index=True)
all_labels_gdf = gpd.GeoDataFrame(all_labels_gdf, crs=all_labels_gdfs[0].crs)

print(f"\nTotal all_labels: {len(all_labels_gdf)} polygons")
print(f"Columns: {all_labels_gdf.columns.tolist()}")
print(f"CRS: {all_labels_gdf.crs}")

In [ ]:
# Load training labels
train_gdf = gpd.read_file(TRAIN_LABELS_PATH)
print(f"Train labels: {len(train_gdf)} polygons")
print(f"Train columns: {train_gdf.columns.tolist()}")
print(f"Train CRS: {train_gdf.crs}")

# train_labels.shp uses 'label' column, but we need 'tree_type' for consistency
# Rename 'label' to 'tree_type' if it exists
if 'label' in train_gdf.columns and 'tree_type' not in train_gdf.columns:
    train_gdf['tree_type'] = train_gdf['label']
    print(f"\nRenamed 'label' column to 'tree_type' for consistency")

# Check tree_type distribution
print(f"\nTree type distribution in train:")
print(train_gdf['tree_type'].value_counts().sort_index())

In [ ]:
# Ensure both GeoDataFrames have the same CRS for spatial operations
if all_labels_gdf.crs != train_gdf.crs:
    print(f"Reprojecting train_gdf from {train_gdf.crs} to {all_labels_gdf.crs}")
    train_gdf = train_gdf.to_crs(all_labels_gdf.crs)

# Remove samples from all_labels that overlap with training samples
# Using spatial join to identify overlapping polygons
print("Removing training samples from all_labels...")

# Find polygons in all_labels that intersect with train polygons
joined = gpd.sjoin(all_labels_gdf, train_gdf, how='left', predicate='intersects')

# Get indices of all_labels polygons that DO intersect with train
intersecting_indices = joined[joined['index_right'].notna()].index.unique()

# Keep only polygons that do NOT intersect with train
all_labels_filtered_gdf = all_labels_gdf[~all_labels_gdf.index.isin(intersecting_indices)].copy()

print(f"\nOriginal all_labels: {len(all_labels_gdf)} polygons")
print(f"Removed (overlapping with train): {len(intersecting_indices)} polygons")
print(f"Filtered all_labels: {len(all_labels_filtered_gdf)} polygons")

# Reset index
all_labels_filtered_gdf = all_labels_filtered_gdf.reset_index(drop=True)

# Show tree_type distribution after filtering
print(f"\nTree type distribution in filtered all_labels:")
print(all_labels_filtered_gdf['tree_type'].value_counts().sort_index())

## 3. Carregar Modelo DeepLabv3+ResNet50

Carregamos o modelo treinado na iter_001 com os pesos do checkpoint.

In [ ]:
# Get image metadata to determine number of channels
sample_ortho_metadata = get_image_metadata(str(ORTHO_PATHS[0]))
IN_CHANNELS = sample_ortho_metadata['count']
print(f"Number of input channels: {IN_CHANNELS}")

# Build model
model = build_model(
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    arch='deeplabv3_resnet50',
    pretrained=False,  # We'll load weights from checkpoint
    psize=INPUT_DIMENSION,
    dropout_rate=DROPOUT_RATE,
    batch_norm=BATCH_NORM,
)

# Load weights from checkpoint
model = load_weights(model, str(CHECKPOINT_PATH))

# Move model to device and set to eval mode
model = model.to(DEVICE)
model.eval()

print(f"\nModel loaded successfully!")
print(f"Architecture: DeepLabv3+ResNet50")
print(f"Input dimension: {INPUT_DIMENSION}x{INPUT_DIMENSION}")
print(f"Number of classes: {NUM_CLASSES}")

## 4. Identificar Região Geográfica de Cada Polígono

Para os dados de treino, precisamos identificar em qual região (ortoimagem) cada polígono está localizado.

In [ ]:
# Load orthoimage metadata and create bounding boxes for each region
ortho_info = []
for region_idx, ortho_path in enumerate(ORTHO_PATHS):
    with rasterio.open(ortho_path) as src:
        bounds = src.bounds
        crs = src.crs
        transform = src.transform
        
        ortho_info.append({
            'region_idx': region_idx,
            'path': ortho_path,
            'bounds': bounds,
            'crs': crs,
            'transform': transform,
            'width': src.width,
            'height': src.height,
        })
        
        print(f"Region {region_idx}: {ortho_path.name}")
        print(f"  CRS: {crs}")
        print(f"  Bounds: {bounds}")
        print(f"  Size: {src.width}x{src.height}")
        print()

In [ ]:
def identify_region_for_polygon(polygon, polygon_crs, ortho_info_list):
    """
    Identify which orthoimage region contains the given polygon.
    
    Parameters
    ----------
    polygon : shapely.geometry
        The polygon geometry
    polygon_crs : CRS
        The CRS of the polygon
    ortho_info_list : list
        List of dicts with orthoimage metadata
    
    Returns
    -------
    int or None
        Region index if found, None otherwise
    """
    centroid = polygon.centroid
    
    for info in ortho_info_list:
        # Create bounding box geometry
        bbox = box(info['bounds'].left, info['bounds'].bottom, 
                   info['bounds'].right, info['bounds'].top)
        
        # Reproject centroid if CRS doesn't match
        if polygon_crs != info['crs']:
            # Create a temporary GeoDataFrame to reproject
            temp_gdf = gpd.GeoDataFrame(geometry=[centroid], crs=polygon_crs)
            temp_gdf = temp_gdf.to_crs(info['crs'])
            centroid_reproj = temp_gdf.geometry.iloc[0]
        else:
            centroid_reproj = centroid
        
        if bbox.contains(centroid_reproj):
            return info['region_idx']
    
    return None

# Assign region_idx to train polygons
print("Identifying regions for train polygons...")
train_region_idx = []
for idx, row in tqdm(train_gdf.iterrows(), total=len(train_gdf)):
    region = identify_region_for_polygon(row.geometry, train_gdf.crs, ortho_info)
    train_region_idx.append(region)

train_gdf['region_idx'] = train_region_idx

# Check for polygons that couldn't be matched
unmatched = train_gdf[train_gdf['region_idx'].isna()]
print(f"\nTrain polygons matched: {len(train_gdf) - len(unmatched)}/{len(train_gdf)}")
if len(unmatched) > 0:
    print(f"Warning: {len(unmatched)} train polygons could not be matched to any region")

# Show distribution
print(f"\nTrain polygons per region:")
print(train_gdf['region_idx'].value_counts().sort_index())

## 5. Funções para Extração de Patches e Máscaras

Funções para:
- Extrair patches 1024x1024 centrados no centroide de cada polígono
- Criar máscaras binárias rasterizando o polígono

In [ ]:
def extract_patch_and_mask(polygon, polygon_crs, region_idx, ortho_info_list, crop_size=1024):
    """
    Extract a patch centered on the polygon centroid and create a binary mask.
    
    Parameters
    ----------
    polygon : shapely.geometry
        The polygon geometry
    polygon_crs : CRS
        The CRS of the polygon
    region_idx : int
        Index of the region (orthoimage) to use
    ortho_info_list : list
        List of dicts with orthoimage metadata
    crop_size : int
        Size of the patch to extract (default: 1024)
    
    Returns
    -------
    tuple
        (patch, mask, valid) where:
        - patch: numpy array of shape (C, H, W)
        - mask: numpy array of shape (H, W) with binary mask
        - valid: bool indicating if extraction was successful
    """
    if region_idx is None or np.isnan(region_idx):
        return None, None, False
    
    region_idx = int(region_idx)
    info = ortho_info_list[region_idx]
    
    # Reproject polygon to orthoimage CRS if needed
    if polygon_crs != info['crs']:
        temp_gdf = gpd.GeoDataFrame(geometry=[polygon], crs=polygon_crs)
        temp_gdf = temp_gdf.to_crs(info['crs'])
        polygon_reproj = temp_gdf.geometry.iloc[0]
    else:
        polygon_reproj = polygon
    
    # Get centroid in pixel coordinates
    centroid = polygon_reproj.centroid
    transform = info['transform']
    
    # Convert centroid to pixel coordinates
    col_center = int((centroid.x - transform.c) / transform.a)
    row_center = int((centroid.y - transform.f) / transform.e)
    
    # Calculate window bounds
    half_size = crop_size // 2
    col_start = col_center - half_size
    row_start = row_center - half_size
    
    # Ensure window is within image bounds
    col_start = max(0, min(col_start, info['width'] - crop_size))
    row_start = max(0, min(row_start, info['height'] - crop_size))
    
    # Create window
    window = Window(col_start, row_start, crop_size, crop_size)
    
    # Calculate transform for the window
    window_transform = rasterio.windows.transform(window, transform)
    
    try:
        with rasterio.open(info['path']) as src:
            # Read patch
            patch = src.read(window=window)
            
            # Handle edge cases where patch might be smaller than crop_size
            if patch.shape[1] != crop_size or patch.shape[2] != crop_size:
                # Pad with zeros
                padded_patch = np.zeros((patch.shape[0], crop_size, crop_size), dtype=patch.dtype)
                padded_patch[:, :patch.shape[1], :patch.shape[2]] = patch
                patch = padded_patch
        
        # Create binary mask by rasterizing the polygon
        mask = rasterize(
            [(polygon_reproj, 1)],
            out_shape=(crop_size, crop_size),
            transform=window_transform,
            fill=0,
            dtype=np.uint8
        )
        
        return patch, mask, True
        
    except Exception as e:
        print(f"Error extracting patch: {e}")
        return None, None, False


# Test the function with one polygon
test_row = all_labels_filtered_gdf.iloc[0]
test_patch, test_mask, test_valid = extract_patch_and_mask(
    test_row.geometry, 
    all_labels_filtered_gdf.crs, 
    test_row['region_idx'],
    ortho_info,
    CROP_SIZE
)

if test_valid:
    print(f"Test extraction successful!")
    print(f"Patch shape: {test_patch.shape}")
    print(f"Mask shape: {test_mask.shape}")
    print(f"Mask coverage: {test_mask.sum()} pixels ({100*test_mask.sum()/(CROP_SIZE*CROP_SIZE):.2f}%)")
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # RGB patch (assuming first 3 channels)
    rgb = np.moveaxis(test_patch[:3], 0, -1)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    axes[0].imshow(rgb)
    axes[0].set_title(f"Patch (tree_type={test_row['tree_type']})")
    
    # Mask
    axes[1].imshow(test_mask, cmap='gray')
    axes[1].set_title("Binary Mask")
    
    # Overlay
    axes[2].imshow(rgb)
    axes[2].imshow(test_mask, alpha=0.3, cmap='Reds')
    axes[2].set_title("Overlay")
    
    plt.tight_layout()
    plt.show()
else:
    print("Test extraction failed!")

## 6. Inferência: Confiança do Modelo e Extração de Features

Para cada patch:
1. **Confiança**: Executar inferência com softmax, extrair probabilidade da classe do polígono e calcular média dentro da máscara
2. **Features**: Extrair features do backbone ResNet50 usando Global Average Pooling

In [ ]:
def extract_backbone_features(model, x):
    """
    Extract features from the ResNet50 backbone (before ASPP).
    
    Parameters
    ----------
    model : DeepLabv3
        The DeepLabv3 model
    x : torch.Tensor
        Input tensor of shape (B, C, H, W)
    
    Returns
    -------
    torch.Tensor
        Feature tensor of shape (B, 2048)
    """
    backbone = model.model.backbone
    
    # Apply batch normalization if the model has it
    if model.batch_norm:
        x = model.batch_norm_layer(x)
    
    # Forward through backbone layers
    x = backbone.conv1(x)
    x = backbone.bn1(x)
    x = backbone.relu(x)
    x = backbone.maxpool(x)
    x = backbone.layer1(x)
    x = backbone.layer2(x)
    x = backbone.layer3(x)
    x = backbone.layer4(x)  # Output shape: (B, 2048, H/32, W/32)
    
    # Global Average Pooling
    features = F.adaptive_avg_pool2d(x, (1, 1))
    
    return features.flatten(1)  # Shape: (B, 2048)


def preprocess_patch(patch, input_dimension):
    """
    Preprocess a patch for model inference.
    
    Parameters
    ----------
    patch : numpy.ndarray
        Patch of shape (C, H, W)
    input_dimension : int
        Target size for model input
    
    Returns
    -------
    torch.Tensor
        Preprocessed tensor of shape (1, C, input_dimension, input_dimension)
    """
    # Convert to float32 and normalize
    patch = patch.astype(np.float32)
    
    # Normalize each channel (in-place)
    for i in range(patch.shape[0]):
        channel = patch[i]
        mean = channel.mean()
        std = channel.std()
        if std > 0:
            patch[i] = (channel - mean) / std
    
    # Convert to tensor
    tensor = torch.from_numpy(patch).unsqueeze(0)  # (1, C, H, W)
    
    # Resize to input_dimension
    if tensor.shape[-1] != input_dimension or tensor.shape[-2] != input_dimension:
        tensor = F.interpolate(
            tensor,
            size=(input_dimension, input_dimension),
            mode='bilinear',
            align_corners=False
        )
    
    return tensor


def compute_confidence_and_features(patch, mask, tree_type, model, input_dimension, crop_size, device):
    """
    Compute mean confidence and backbone features for a patch.
    
    Parameters
    ----------
    patch : numpy.ndarray
        Patch of shape (C, H, W)
    mask : numpy.ndarray
        Binary mask of shape (H, W)
    tree_type : int
        Class of the polygon (1-4)
    model : nn.Module
        The DeepLabv3 model
    input_dimension : int
        Size of model input
    crop_size : int
        Original crop size
    device : torch.device
        Device to run inference on
    
    Returns
    -------
    tuple
        (mean_confidence, features) where:
        - mean_confidence: float, mean probability within mask
        - features: numpy array of shape (2048,)
    """
    # Preprocess
    tensor = preprocess_patch(patch, input_dimension)
    tensor = tensor.to(device, dtype=torch.float)
    
    with torch.no_grad():
        # Get model output
        out = model(tensor)
        
        # Apply softmax to get probabilities
        prob_map = F.softmax(out['out'], dim=1)  # (1, num_classes, H, W)
        
        # Resize probability map back to crop_size
        prob_map = F.interpolate(
            prob_map,
            size=(crop_size, crop_size),
            mode='bilinear',
            align_corners=False
        )
        
        # Get probability for the specific class (tree_type - 1 for 0-indexed)
        class_idx = tree_type - 1
        class_prob = prob_map[0, class_idx].cpu().numpy()  # (H, W)
        
        # Compute mean confidence within mask
        mask_bool = mask.astype(bool)
        if mask_bool.sum() > 0:
            mean_confidence = class_prob[mask_bool].mean()
        else:
            mean_confidence = 0.0
        
        # Extract backbone features
        features = extract_backbone_features(model, tensor)
        features = features.cpu().numpy().flatten()
    
    return mean_confidence, features


# Test the functions
print("Testing confidence and feature extraction...")
test_conf, test_feat = compute_confidence_and_features(
    test_patch, test_mask, test_row['tree_type'],
    model, INPUT_DIMENSION, CROP_SIZE, DEVICE
)
print(f"Mean confidence: {test_conf:.4f}")
print(f"Features shape: {test_feat.shape}")
print(f"Features range: [{test_feat.min():.4f}, {test_feat.max():.4f}]")

## 7. Processar Amostras de Treino

Primeiro, extraímos features de todas as amostras de treino para usar como referência no cálculo de distância.

In [ ]:
# Filter train samples that have a valid region assignment
train_valid_gdf = train_gdf[train_gdf['region_idx'].notna()].copy()
print(f"Processing {len(train_valid_gdf)} train samples with valid region assignment")

# Extract features for all training samples
train_features_dict = {tree_type: [] for tree_type in range(1, NUM_CLASSES + 1)}

print("\nExtracting features from training samples...")
for idx, row in tqdm(train_valid_gdf.iterrows(), total=len(train_valid_gdf)):
    patch, mask, valid = extract_patch_and_mask(
        row.geometry, train_gdf.crs, row['region_idx'], ortho_info, CROP_SIZE
    )
    
    if not valid:
        continue
    
    _, features = compute_confidence_and_features(
        patch, mask, row['tree_type'], model, INPUT_DIMENSION, CROP_SIZE, DEVICE
    )
    
    train_features_dict[row['tree_type']].append(features)

# Convert to numpy arrays
for tree_type in train_features_dict:
    if train_features_dict[tree_type]:
        train_features_dict[tree_type] = np.array(train_features_dict[tree_type])
    else:
        train_features_dict[tree_type] = np.array([]).reshape(0, 2048)

print("\nTraining features extracted:")
for tree_type, features in train_features_dict.items():
    print(f"  Tree type {tree_type}: {len(features)} samples")

## 8. Processar Amostras all_labels e Calcular Métricas

Para cada amostra em all_labels (filtrado):
1. Extrair patch e máscara
2. Calcular confiança média do modelo
3. Extrair features do backbone
4. Calcular distância de cosseno média com amostras de treino da mesma espécie

In [ ]:
def mean_cosine_distance(query_feature, train_features):
    """
    Calculate mean cosine distance between a query feature and training features.
    
    Parameters
    ----------
    query_feature : numpy.ndarray
        Query feature vector of shape (2048,)
    train_features : numpy.ndarray
        Training features of shape (N, 2048)
    
    Returns
    -------
    float
        Mean cosine distance
    """
    if len(train_features) == 0:
        return np.nan
    
    distances = [cosine(query_feature, tf) for tf in train_features]
    return np.mean(distances)


# Process all_labels samples
print(f"Processing {len(all_labels_filtered_gdf)} samples from all_labels...")

results = []

for idx, row in tqdm(all_labels_filtered_gdf.iterrows(), total=len(all_labels_filtered_gdf)):
    # Extract patch and mask
    patch, mask, valid = extract_patch_and_mask(
        row.geometry, all_labels_filtered_gdf.crs, row['region_idx'], ortho_info, CROP_SIZE
    )
    
    if not valid:
        continue
    
    # Compute confidence and features
    mean_confidence, features = compute_confidence_and_features(
        patch, mask, row['tree_type'], model, INPUT_DIMENSION, CROP_SIZE, DEVICE
    )
    
    # Calculate mean cosine distance to training samples of same class
    tree_type = row['tree_type']
    train_feats = train_features_dict.get(tree_type, np.array([]).reshape(0, 2048))
    mean_cosine_dist = mean_cosine_distance(features, train_feats)
    
    results.append({
        'polygon_id': idx,
        'tree_type': tree_type,
        'region_idx': row['region_idx'],
        'mean_confidence': mean_confidence,
        'mean_cosine_distance': mean_cosine_dist,
        'mask_coverage': mask.sum() / (CROP_SIZE * CROP_SIZE),
    })

# Create results DataFrame
results_df = pd.DataFrame(results)
print(f"\nResults: {len(results_df)} samples processed successfully")
print(results_df.head(10))

## 9. Análise de Correlação

Analisamos a correlação entre:
- **Confiança média do modelo** (`mean_confidence`)
- **Distância de cosseno média** (`mean_cosine_distance`)

A hipótese é que amostras com **maior confiança** devem ter **menor distância** no espaço de features (ou seja, correlação negativa).

In [ ]:
# Remove rows with NaN values
results_clean = results_df.dropna(subset=['mean_confidence', 'mean_cosine_distance'])
print(f"Samples for analysis: {len(results_clean)} (dropped {len(results_df) - len(results_clean)} with NaN)")

# Basic statistics
print("\n=== Basic Statistics ===")
print(results_clean[['mean_confidence', 'mean_cosine_distance']].describe())

In [ ]:
# Calculate correlations
confidence = results_clean['mean_confidence'].values
cosine_dist = results_clean['mean_cosine_distance'].values

# Pearson correlation
pearson_corr, pearson_p = pearsonr(confidence, cosine_dist)
print("=== Global Correlation ===")
print(f"Pearson correlation:  r = {pearson_corr:.4f}, p-value = {pearson_p:.2e}")

# Spearman correlation (rank-based, more robust)
spearman_corr, spearman_p = spearmanr(confidence, cosine_dist)
print(f"Spearman correlation: ρ = {spearman_corr:.4f}, p-value = {spearman_p:.2e}")

In [ ]:
# Scatter + density plot: Confidence vs Cosine Distance using seaborn

import seaborn as sns

plt.figure(figsize=(10, 8))

# Use scatterplot with KDE overlay
scatter = sns.scatterplot(
    data=results_clean, 
    x='mean_confidence', 
    y='mean_cosine_distance', 
    hue='tree_type',
    palette='viridis',
    alpha=0.6,
    edgecolor='white',
    linewidth=0.5,
    legend="full"
)

# Add 2D KDE contours
sns.kdeplot(
    data=results_clean,
    x='mean_confidence',
    y='mean_cosine_distance',
    ax=scatter,
    levels=5,
    color='gray',
    linewidths=1,
    alpha=0.5
)

# Add regression line
z = np.polyfit(confidence, cosine_dist, 1)
p = np.poly1d(z)
x_line = np.linspace(confidence.min(), confidence.max(), 100)
scatter.plot(x_line, p(x_line), "r--", linewidth=2, label=f'Linear fit (r={pearson_corr:.3f})')

scatter.set_xlabel('Mean Confidence', fontsize=12)
scatter.set_ylabel('Mean Cosine Distance', fontsize=12)
scatter.set_title(f'Confidence vs Feature Space Distance\n(Pearson r={pearson_corr:.3f}, Spearman ρ={spearman_corr:.3f})', fontsize=14)

# Add legend for tree_type
handles, labels = scatter.get_legend_handles_labels()
scatter.legend(handles=handles, labels=labels, title='Tree Type', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class correlation analysis
print("=== Per-Class Correlation ===\n")

class_correlations = []
for tree_type in sorted(results_clean['tree_type'].unique()):
    subset = results_clean[results_clean['tree_type'] == tree_type]
    
    if len(subset) < 3:
        print(f"Tree type {tree_type}: Insufficient samples ({len(subset)})")
        continue
    
    conf = subset['mean_confidence'].values
    dist = subset['mean_cosine_distance'].values
    
    try:
        p_corr, p_pval = pearsonr(conf, dist)
        s_corr, s_pval = spearmanr(conf, dist)
    except Exception as e:
        print(f"Tree type {tree_type}: Error computing correlation - {e}")
        continue
    
    class_correlations.append({
        'tree_type': tree_type,
        'n_samples': len(subset),
        'pearson_r': p_corr,
        'pearson_p': p_pval,
        'spearman_r': s_corr,
        'spearman_p': s_pval,
    })
    
    print(f"Tree type {tree_type} (n={len(subset)}):")
    print(f"  Pearson:  r = {p_corr:.4f}, p = {p_pval:.2e}")
    print(f"  Spearman: ρ = {s_corr:.4f}, p = {s_pval:.2e}")
    print()

# Summary DataFrame
class_corr_df = pd.DataFrame(class_correlations)
if len(class_corr_df) > 0:
    print("=== Summary Table ===")
    display(class_corr_df)

In [ ]:
# Per-class scatter plots
tree_types = sorted(results_clean['tree_type'].unique())
n_classes = len(tree_types)

fig, axes = plt.subplots(1, n_classes, figsize=(5*n_classes, 5))
if n_classes == 1:
    axes = [axes]

for ax, tree_type in zip(axes, tree_types):
    subset = results_clean[results_clean['tree_type'] == tree_type]
    
    if len(subset) < 2:
        ax.set_title(f'Tree Type {tree_type}\n(n={len(subset)})')
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)
        continue
    
    ax.scatter(
        subset['mean_confidence'], 
        subset['mean_cosine_distance'],
        alpha=0.6,
        edgecolors='white',
        linewidths=0.5
    )
    
    # Add regression line
    conf = subset['mean_confidence'].values
    dist = subset['mean_cosine_distance'].values
    
    if len(subset) >= 3:
        try:
            z = np.polyfit(conf, dist, 1)
            p = np.poly1d(z)
            x_line = np.linspace(conf.min(), conf.max(), 100)
            ax.plot(x_line, p(x_line), "r--", linewidth=2)
            
            r, _ = pearsonr(conf, dist)
            ax.set_title(f'Tree Type {tree_type}\n(n={len(subset)}, r={r:.3f})')
        except:
            ax.set_title(f'Tree Type {tree_type}\n(n={len(subset)})')
    else:
        ax.set_title(f'Tree Type {tree_type}\n(n={len(subset)})')
    
    ax.set_xlabel('Mean Confidence')
    ax.set_ylabel('Mean Cosine Distance')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Confidence distribution
axes[0].hist(results_clean['mean_confidence'], bins=30, edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Mean Confidence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Model Confidence')
axes[0].axvline(results_clean['mean_confidence'].mean(), color='red', linestyle='--', label=f"Mean: {results_clean['mean_confidence'].mean():.3f}")
axes[0].legend()

# Cosine distance distribution
axes[1].hist(results_clean['mean_cosine_distance'], bins=30, edgecolor='white', alpha=0.7)
axes[1].set_xlabel('Mean Cosine Distance')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Feature Distance')
axes[1].axvline(results_clean['mean_cosine_distance'].mean(), color='red', linestyle='--', label=f"Mean: {results_clean['mean_cosine_distance'].mean():.3f}")
axes[1].legend()

# Box plot by tree type
results_clean.boxplot(column='mean_cosine_distance', by='tree_type', ax=axes[2])
axes[2].set_xlabel('Tree Type')
axes[2].set_ylabel('Mean Cosine Distance')
axes[2].set_title('Cosine Distance by Tree Type')
plt.suptitle('')  # Remove automatic suptitle from boxplot

plt.tight_layout()
plt.show()

## 10. Conclusões

### Interpretação dos Resultados

- **Correlação negativa**: Se a correlação for negativa, isso indica que amostras com **maior confiança** do modelo tendem a estar **mais próximas** das amostras de treino no espaço de features. Isso suporta a hipótese de que o modelo é mais confiante em amostras similares ao conjunto de treino.

- **Correlação positiva**: Se a correlação for positiva, isso sugere que amostras com maior confiança estão mais distantes das amostras de treino, o que pode indicar overfitting ou viés do modelo.

- **Sem correlação significativa**: Se não houver correlação significativa, a confiança do modelo pode não estar diretamente relacionada à similaridade com o conjunto de treino no espaço de features.

In [ ]:
# Final Summary
print("=" * 60)
print("SUMMARY: Confidence vs Feature Space Distance Analysis")
print("=" * 60)
print(f"\nTotal samples analyzed: {len(results_clean)}")
print(f"Training samples used as reference: {sum(len(v) for v in train_features_dict.values())}")
print(f"\nGlobal Correlation:")
print(f"  Pearson r:  {pearson_corr:.4f} (p={pearson_p:.2e})")
print(f"  Spearman ρ: {spearman_corr:.4f} (p={spearman_p:.2e})")

# Interpretation
print(f"\nInterpretation:")
if pearson_p < 0.05:
    if pearson_corr < -0.3:
        print("  Strong negative correlation: Higher confidence → Closer to training samples")
    elif pearson_corr < 0:
        print("  Weak negative correlation: Slight tendency for higher confidence → Closer to training")
    elif pearson_corr > 0.3:
        print("  Strong positive correlation: Higher confidence → Further from training samples")
    else:
        print("  Weak positive correlation: Slight tendency for higher confidence → Further from training")
else:
    print("  No statistically significant correlation found")

print("\n" + "=" * 60)

# Save results to CSV
output_path = PROJECT_ROOT / "exploration_notebooks" / "bioflore" / "confidence_feature_correlation_results.csv"
results_clean.to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path}")